# Comparación de métodos de detección de agua para el *water frequency* y el intermareal

**Objetivo.** Decidir **qué criterio de detección de agua funciona mejor cuando se inserta en el pipeline** de mapeo intermareal (Ría de Foz). Se comparan cuatro métodos, cada uno con sus propios umbrales:

| Método | Cómo decide "agua" |
|---|---|
| **SCL**  | clase de agua de la *Scene Classification Layer* de Sentinel-2 (6 = agua, 12 = veg. inundada) |
| **NDWI** | `(B03 − B08)/(B03 + B08) > umbral`  (McFeeters 1996) |
| **MNDWI**| `(B03 − B11)/(B03 + B11) > umbral`  (Xu 2006) |
| **AWEI** | `4·(B03 − B11) − (0.25·B08 + 2.75·B12) > umbral`  (Feyisa 2014, AWEI_nsh) |

**Comparación justa.** Los cuatro se calculan en **un solo job de OpenEO**, sobre **las mismas fechas** y con **la misma máscara de nubes SCL** en el denominador. Lo único que cambia entre métodos es el criterio agua/no-agua.

**Fidelidad al pipeline.** El intermareal se deriva igual que en `final_notebook` (celda 17): un píxel es intermareal si está en la **zona de transición** del *reference map* (SCL) **y** su *water frequency* cae en `[low, high]`. El reference map se mantiene SCL (fijo) para todos → se aísla el efecto del método de detección de agua.

**Validación independiente.** Se descarga el **MDT LiDAR del IGN** (5 m). En el intermareal, la frecuencia de agua debe **decrecer con la elevación**; el método cuyo WF mejor correlaciona (Spearman más negativo) con la cota real es el más consistente físicamente. Métrica **sin umbrales arbitrarios**.

> Toda la lógica reutilizable está en el paquete (`intertidal.notebook_compat`, `intertidal.validation`); este notebook solo **orquesta y pinta**.
>
> **Reutilizable:** cambia `site`, `bbox`, `time_extent` (y opcionalmente los umbrales) para otro estuario.

## 0 · Imports

`truststore` hace que OpenEO/`requests` validen el SSL con el almacén de certificados de Windows. La lógica de comparación/validación vive en `intertidal.validation`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from rasterio.warp import Resampling
try:
    import truststore; truststore.inject_into_ssl()
except Exception as e:
    print("truststore no disponible:", e)
import openeo

from intertidal.notebook_compat import compute_water_frequency_multi_openeo
from intertidal.validation import (
    download_mdt_ign, reproject_to_grid,
    validate_wf_vs_elevation, intertidal_from_wf, pairwise_iou,
)

## 1 · Configuración

- `bbox`/`time_extent`: área y periodo (mismos que el pipeline principal de Foz).
- `methods` + `thresholds`: métodos a comparar y su umbral de índice (SCL no usa umbral: usa clases).
- `clear_classes`: clases SCL que cuentan como observación clara (denominador del WF) — **igual para todos**.
- `low_wf`/`high_wf`: banda de *water frequency* que define el intermareal (misma que el pipeline).

In [ ]:
site = "Foz"
# bbox en EPSG:32629 (mismo grid 10 m que el pipeline principal de Foz)
bbox = {"west": 638440, "south": 4819990, "east": 645070, "north": 4827910, "crs": "EPSG:32629"}
time_extent = ["2023-01-01", "2023-12-31"]

# Métodos a comparar y umbral de detección de agua de cada uno.
# (SCL define agua por clases; NDWI/MNDWI/AWEI: índice > umbral. Ajusta aquí
#  para optimizar cada método por separado.)
methods    = ["scl", "ndwi", "mndwi", "awei"]
thresholds = {"ndwi": 0.0, "mndwi": 0.0, "awei": 0.0}
scl_water_classes = [6, 12]        # clases SCL consideradas agua
clear_classes     = [4, 5, 6, 12]  # clases SCL consideradas observación clara (sin nube)

min_obs = 8            # mínimo de observaciones claras por píxel; por debajo -> NaN
max_cloud_cover = 40   # filtro de escena a la carga

# Banda intermareal sobre el water frequency (idéntica al pipeline, celda 17)
low_wf, high_wf = 0.15, 0.85

pixel_m = 10.0
px_km2  = pixel_m**2 / 1e6
wf_path  = f"water_frequency_multi_{site.lower()}.tif"
mdt_path = f"mdt5_{site.lower()}.tif"
# Reference map (SCL) del pipeline: 0 = transición (candidata a intermareal).
ref_path = "reference_map_openeo.tif"

## 2 · Conexión a OpenEO

Requiere estar autenticado (usa el token cacheado si ya lo estabas).

In [ ]:
conn = openeo.connect("openeo.dataspace.copernicus.eu")
conn.authenticate_oidc()
print("OpenEO conectado")

## 3 · Water frequency multi-método (un solo job)

`compute_water_frequency_multi_openeo` baja las bandas **una sola vez** y calcula, en el backend, el WF de cada método:

`WF_método = (nº fechas con AGUA según el método) / (nº fechas con observación CLARA según SCL)`

El denominador (nubes) es el mismo para todos → comparación limpia. Con `force=False` reutiliza el `.tif` si ya existe.

In [ ]:
wf, wf_transform, wf_crs = compute_water_frequency_multi_openeo(
    conn, bbox, time_extent,
    methods=tuple(methods), thresholds=thresholds,
    clear_classes=tuple(clear_classes), scl_water_classes=tuple(scl_water_classes),
    min_obs=min_obs, max_cloud_cover=max_cloud_cover,
    out_path=wf_path, force=False,   # force=True para recalcular
)
shp = next(iter(wf.values())).shape
print("métodos:", list(wf.keys()), "| shape:", shp)

fig, axes = plt.subplots(1, len(methods), figsize=(5*len(methods), 5))
for ax, m in zip(np.atleast_1d(axes), methods):
    im = ax.imshow(wf[m], cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"WF | {m.upper()}"); ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.6, label="Frecuencia de agua")
plt.show()

## 4 · Zona de transición (pipeline) + referencia LiDAR

Dos rásters auxiliares, reproyectados al grid del WF (funciones del paquete `intertidal.validation`):

1. **Reference map (SCL)** → `transition = (refmap == 0)`: zona candidata a intermareal del pipeline.
2. **MDT LiDAR del IGN (5 m)** vía WCS → referencia de elevación independiente. Limitación: el LiDAR **recorta el agua a 0 m** → valida los **flats expuestos**, no la parte sumergida.

In [ ]:
# Zona de transición (reference map SCL) reproyectada al grid del WF
if os.path.exists(ref_path):
    refmap = np.round(reproject_to_grid(ref_path, wf_transform, wf_crs, shp, Resampling.nearest))
    transition = (refmap == 0)
    print(f"Reference map | zona de transición: {int(transition.sum()):,} px")
else:
    transition = np.ones(shp, bool)
    print("AVISO: sin reference_map_openeo.tif -> comparación SIN restricción de transición")

# MDT LiDAR de referencia (descarga si no existe) y reproyección al grid
download_mdt_ign(bbox, mdt_path)          # no re-descarga si ya existe
mdt = reproject_to_grid(mdt_path, wf_transform, wf_crs, shp)
print(f"MDT LiDAR | elevación {np.nanmin(mdt):.1f} -> {np.nanmax(mdt):.1f} m")

## 5 · Validación objetiva: WF ↔ elevación LiDAR

En la franja intermareal el WF debe **decrecer con la cota**. `validate_wf_vs_elevation` correlaciona WF con la elevación LiDAR en **transición ∩ flats expuestos** (`0.3 < z ≤ 5 m`). **Spearman más negativo = mejor.**

In [ ]:
z_range = (0.3, 5.0)
val = validate_wf_vs_elevation(wf, mdt, transition, z_range=z_range)
scores = {m: val[m]["spearman"] for m in methods}

print(f"Píxeles de validación (transición + flats expuestos): {val[methods[0]]['n']:,}")
print()
print(f"{'método':>7} | {'Spearman':>9} | {'Pearson':>8}")
for m in methods:
    print(f"{m:>7} | {val[m]['spearman']:>9.3f} | {val[m]['pearson']:>8.3f}")
best = min(scores, key=scores.get)
print()
print(f"=> mejor consistencia con el LiDAR (más negativo): {best.upper()} ({scores[best]:.3f})")

### Curvas frecuencia–elevación (hipsométricas)

WF medio por cota. La curva ideal **decrece de forma monótona y limpia**.

In [ ]:
col = {"scl":"#333333","ndwi":"#1f77b4","mndwi":"#d62728","awei":"#2ca02c"}
z_lo, z_hi = z_range
valid_zone = transition & np.isfinite(mdt) & (mdt > z_lo) & (mdt <= z_hi)
bins = np.linspace(z_lo, z_hi, 20); cen = 0.5*(bins[:-1]+bins[1:])
fig, ax = plt.subplots(figsize=(9, 6))
for m in methods:
    ys = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        sel = valid_zone & np.isfinite(wf[m]) & (mdt >= lo) & (mdt < hi)
        ys.append(np.nanmean(wf[m][sel]) if sel.sum() > 20 else np.nan)
    ax.plot(cen, ys, "-o", ms=4, color=col.get(m), label=f"{m.upper()} (rho={scores[m]:.2f})")
ax.set_xlabel("Elevación LiDAR IGN (m)"); ax.set_ylabel("Water frequency media")
ax.set_title(f"Curva frecuencia-elevación validada con LiDAR - {site}", fontweight="bold")
ax.grid(alpha=0.3); ax.legend(); plt.show()

## 6 · Intermareal por método (idéntico al pipeline)

`intertidal_from_wf` construye la máscara como el pipeline (celda 17): **transición** ∩ `WF ∈ [low, high]`. Se compara el **área** y el **solape (IoU)** entre métodos.

In [ ]:
inter = intertidal_from_wf(wf, transition, low_wf, high_wf)

print(f"Área intermareal (transición & WF ∈ [{low_wf}, {high_wf}]):")
for m in methods:
    print(f"  {m:>6}: {inter[m].sum()*px_km2:6.2f} km²")

print()
print("IoU (solape) entre métodos:")
for (a, b), iou in pairwise_iou(inter).items():
    print(f"  {a:>5}-{b:<5}: {iou:.2f}")

fig, axes = plt.subplots(1, len(methods), figsize=(5*len(methods), 5))
for ax, m in zip(np.atleast_1d(axes), methods):
    ax.imshow(inter[m], cmap="Blues")
    ax.set_title(f"Intertidal | {m.upper()} ({inter[m].sum()*px_km2:.2f} km2)"); ax.axis("off")
plt.show()

## 7 · Cómo leerlo y siguientes pasos

- **Ganador = Spearman más negativo** (sección 5): mejor coherencia con la topografía LiDAR real. El área (sección 6) muestra cómo de amplio delimita cada método.
- **Optimizar umbrales (mejor-vs-mejor):** cada método tiene su umbral en `thresholds` (SCL vía clases). Cámbialos, re-ejecuta la sección 3 (con `force=True`) y vuelve a validar. La comparación justa es cada método en su mejor umbral.
- **Multi-sitio:** repite con otro `site`/`bbox`/`time_extent` (p.ej. un estuario menos turbio) para ver si el ranking se mantiene — clave para escalar el pipeline.

**Limitaciones:**
- El MDT LiDAR **recorta el agua a 0 m** → valida los flats expuestos, no la parte sumergida.
- El WF multi usa todas las escenas con nube < `max_cloud_cover`, no los `valid_dates` exactos del filtrado de nubes en transición. Para todos los métodos es la misma base (comparación justa); para replicar el pipeline al 100% habría que pasar `valid_dates`.